# Mount Drive và tạo thư mục

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/datn'
RAW_DRIVE = f'{BASE}/raw'          # bản nén .zip của từng bộ (lưu lâu dài)
RAW_LOCAL = '/content/raw'         # bản giải nén trên máy Colab (đọc nhanh)
for d in ['raw', 'merged', 'runs', 'own_images']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)
os.makedirs(RAW_LOCAL, exist_ok=True)
print('OK:', BASE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OK: /content/drive/MyDrive/datn


# Cài thư viện

In [ ]:
!pip install -q roboflow imagehash pyyaml

# Lấy API key


In [ ]:
from google.colab import userdata
from roboflow import Roboflow

API_KEY = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=API_KEY)
print('Đã kết nối Roboflow')

Đã kết nối Roboflow


## Khai báo 4 bộ dataset
Danh sách 4 dataset trên Roboflow Universe.

| Tên ngắn | Link |
|---|---|
| super | https://universe.roboflow.com/312s-workspace/xiangqi-super-dataset |
| scara | https://universe.roboflow.com/312s-workspace/xiangqi-scara-yolo26 |
| new_chinese_chess | https://universe.roboflow.com/quoc-rnmzr/new-chinese-chess-ayph4/dataset/1 |
| duytranle | https://universe.roboflow.com/duytranle/xiangqi-ubjij/dataset/18 |

In [ ]:
DATASETS = {
    'super':             {'workspace': '312s-workspace', 'project': 'xiangqi-super-dataset',   'version': None},
    'scara':             {'workspace': '312s-workspace', 'project': 'xiangqi-scara-yolo26',    'version': None},
    'new_chinese_chess': {'workspace': 'quoc-rnmzr',     'project': 'new-chinese-chess-ayph4', 'version': 1},
    'duytranle':         {'workspace': 'duytranle',      'project': 'xiangqi-ubjij',           'version': 18,
                          'format': 'yolov8'},   # dự án keypoint không hỗ trợ yolov11
}

### Tải về, giải nén và sao lưu lên Drive
Với mỗi bộ: (1) đã có trên Colab thì bỏ qua; (2) đã có file .zip trên Drive thì giải nén; (3) chưa có thì tải từ Roboflow (định dạng yolov11) rồi nén lưu lên Drive. Phiên bản đã tải được ghi vào versions_used.json để đưa vào báo cáo.

In [ ]:
import shutil, json

def latest_version(project):
    vs = project.versions()
    return max(int(str(v.version).split('/')[-1]) for v in vs)

VERSIONS_FILE = f'{BASE}/raw/versions_used.json'
versions_used = json.load(open(VERSIONS_FILE)) if os.path.exists(VERSIONS_FILE) else {}
paths = {}

for name, cfg in DATASETS.items():
    out_dir  = f'{RAW_LOCAL}/{name}'
    zip_base = f'{RAW_DRIVE}/{name}'

    if os.path.isdir(out_dir) and os.listdir(out_dir):
        print(f'[{name}] đã có trên Colab, bỏ qua')
    elif os.path.exists(zip_base + '.zip'):
        print(f'[{name}] giải nén từ Drive...')
        shutil.unpack_archive(zip_base + '.zip', out_dir)
    else:
        print(f'[{name}] tải từ Roboflow...')
        project = rf.workspace(cfg['workspace']).project(cfg['project'])
        ver = cfg['version'] or latest_version(project)
        fmt = cfg.get('format', 'yolov11')
        project.version(ver).download(fmt, location=out_dir)
        shutil.make_archive(zip_base, 'zip', out_dir)
        versions_used[name] = {**cfg, 'version': ver, 'format': fmt}
        with open(VERSIONS_FILE, 'w') as f:
            json.dump(versions_used, f, indent=2, ensure_ascii=False)
        print(f'[{name}] đã tải version {ver} (định dạng {fmt}) và lưu lên Drive')

    paths[name] = out_dir

print('Xong:', list(paths))

[super] đã có trên Colab, bỏ qua
[scara] đã có trên Colab, bỏ qua
[new_chinese_chess] đã có trên Colab, bỏ qua
[duytranle] đã có trên Colab, bỏ qua
Xong: ['super', 'scara', 'new_chinese_chess', 'duytranle']


### Kiểm tra tên lớp và số ảnh của từng bộ
In danh sách lớp (names trong data.yaml) và số ảnh mỗi tập. Với mỗi bộ: in tên lớp, số ảnh (và số ảnh gốc ước lượng) mỗi tập, bảng số box theo lớp ở từng tập, và các lỗi nhãn.

In [ ]:
import yaml, glob, os
import pandas as pd
from collections import Counter

SPLITS = ['train', 'valid', 'test']
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
MIN_BOX_WARN = 300          # lớp có ít box hơn mức này sẽ bị cảnh báo
pd.set_option('display.width', 220)

def load_names(yaml_path):
    with open(yaml_path, encoding='utf-8') as f:
        names = yaml.safe_load(f)['names']
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names)]
    return names

def source_id(filename):
    # Roboflow: <tên gốc>.rf.<hash>.jpg -> bản augment cùng một ảnh gốc
    return filename.split('.rf.')[0]

reports = {}
for name, p in paths.items():
    print('=' * 80)
    print(name.upper())
    yaml_files = glob.glob(f'{p}/**/data.yaml', recursive=True)
    if not yaml_files:
        print('  Không thấy data.yaml')
        continue
    root = os.path.dirname(yaml_files[0])
    names = load_names(yaml_files[0])
    print(f'  Số lớp: {len(names)}')
    print(f'  Tên lớp: {names}')

    box = {s: Counter() for s in SPLITS}      # số box theo lớp
    img_with = Counter()                      # số ảnh có chứa lớp (cả 3 tập)
    problems = Counter()

    for s in SPLITS:
        img_files = [f for f in glob.glob(f'{root}/{s}/images/*')
                     if f.lower().endswith(IMG_EXT)]
        n_src = len({source_id(os.path.basename(f)) for f in img_files})
        print(f'  {s:6s}: {len(img_files):5d} ảnh  (~{n_src} ảnh gốc ước lượng)')

        lbl_files = glob.glob(f'{root}/{s}/labels/*.txt')
        img_stems = {os.path.splitext(os.path.basename(f))[0] for f in img_files}
        lbl_stems = {os.path.splitext(os.path.basename(f))[0] for f in lbl_files}
        problems['ảnh không có file nhãn'] += len(img_stems - lbl_stems)
        problems['file nhãn không có ảnh'] += len(lbl_stems - img_stems)

        for lf in lbl_files:
            with open(lf) as f:
                lines = [l.split() for l in f if l.strip()]
            if not lines:
                problems['file nhãn rỗng'] += 1
            seen = set()
            for parts in lines:
                if len(parts) != 5:
                    problems['dòng nhãn không phải bbox (5 giá trị)'] += 1
                    continue
                try:
                    cid = int(parts[0])
                    x, y, w, h = map(float, parts[1:])
                except ValueError:
                    problems['dòng nhãn sai định dạng'] += 1
                    continue
                if not (0 <= cid < len(names)):
                    problems['id lớp ngoài phạm vi'] += 1
                    continue
                if w <= 0 or h <= 0:
                    problems['box có w hoặc h <= 0'] += 1
                if x - w/2 < -0.01 or y - h/2 < -0.01 or x + w/2 > 1.01 or y + h/2 > 1.01:
                    problems['box lệch ra ngoài khung ảnh'] += 1
                box[s][names[cid]] += 1
                seen.add(names[cid])
            for c in seen:
                img_with[c] += 1

    df = pd.DataFrame({f'{s}_box': pd.Series(box[s]) for s in SPLITS}).reindex(names).fillna(0).astype(int)
    df['tong_box'] = df.sum(axis=1)
    df['tong_anh_co_lop'] = pd.Series(img_with).reindex(names).fillna(0).astype(int)
    reports[name] = df

    print('\n  Số box theo lớp:')
    print(df.to_string())
    print(f"\n  Tổng box: {df['tong_box'].sum()}")
    weak = df[df['tong_box'] < MIN_BOX_WARN].index.tolist()
    if weak:
        print(f'  ⚠ Lớp ít hơn {MIN_BOX_WARN} box: {weak}')
    if problems:
        print('  ⚠ Vấn đề phát hiện:')
        for k, v in problems.items():
            print(f'     - {k}: {v}')
    else:
        print('  Không phát hiện lỗi nhãn cơ bản.')

SUPER
  Số lớp: 15
  Tên lớp: ['black_advisor', 'black_bishop', 'black_cannon', 'black_king', 'black_knight', 'black_pawn', 'black_rook', 'board', 'red_advisor', 'red_bishop', 'red_cannon', 'red_king', 'red_knight', 'red_pawn', 'red_rook']
  train :  4107 ảnh  (~1779 ảnh gốc ước lượng)
  valid :   291 ảnh  (~289 ảnh gốc ước lượng)
  test  :    57 ảnh  (~57 ảnh gốc ước lượng)

  Số box theo lớp:
               train_box  valid_box  test_box  tong_box  tong_anh_co_lop
black_advisor       2406        254       101      2761             1461
black_bishop        2108        239        96      2443             1354
black_cannon        1159        112        55      1326              889
black_king          1316        142        52      1510             1504
black_knight        1566        173        63      1802             1256
black_pawn          4387        493       174      5054             1451
black_rook          2310        267        83      2660             1368
board             

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


### Chẩn đoán dòng nhãn không phải bbox
Với mỗi bộ: đếm dòng 5 giá trị (bbox chuẩn) và các dòng khác, gom theo (lớp, số giá trị mỗi dòng), in kèm một dòng mẫu. Dùng để biết dòng lạ là polygon (segmentation) hay pose (keypoint) và thuộc lớp nào.

In [ ]:
from collections import Counter

for name, p in paths.items():
    yf = glob.glob(f'{p}/**/data.yaml', recursive=True)[0]
    root = os.path.dirname(yf)
    names = load_names(yf)
    odd, sample, ok = Counter(), {}, 0
    for lf in glob.glob(f'{root}/*/labels/*.txt'):
        for line in open(lf):
            parts = line.split()
            if not parts:
                continue
            if len(parts) == 5:
                ok += 1
                continue
            key = (names[int(parts[0])], len(parts))
            odd[key] += 1
            sample.setdefault(key, ' '.join(parts[:12]))
    print('=' * 70)
    print(f'{name} | dòng 5 giá trị: {ok} | dòng khác: {sum(odd.values())}')
    for (cls, n), c in odd.most_common(6):
        print(f'  lớp={cls:15s} số giá trị/dòng={n:3d}  số dòng={c}')
        print('     mẫu:', sample[(cls, n)])

super | dòng 5 giá trị: 34886 | dòng khác: 3551
  lớp=board           số giá trị/dòng= 11  số dòng=2137
     mẫu: 7 0.92248125 0.066184375 0.0808421875 0.066184375 0.0808421875 0.937953125 0.92248125 0.937953125 0.92248125 0.066184375
  lớp=red_pawn        số giá trị/dòng= 41  số dòng=80
     mẫu: 13 0.484375 0.6921875 0.46875 0.6984375 0.465625 0.703125 0.4609375 0.7046875 0.45 0.7203125 0.4484375
  lớp=black_pawn      số giá trị/dòng= 43  số dòng=42
     mẫu: 5 0.575 0.30625 0.5625 0.321875 0.5578125 0.3359375 0.5578125 0.346875 0.5609375 0.35625 0.5703125
  lớp=red_pawn        số giá trị/dòng= 39  số dòng=42
     mẫu: 13 0.03125 0.3046875 0.0171875 0.31875 0.0125 0.3328125 0.0125 0.353125 0.01875 0.3671875 0.034375
  lớp=red_pawn        số giá trị/dòng= 47  số dòng=40
     mẫu: 13 0.578125 0.6 0.5671875 0.609375 0.5671875 0.6125 0.5609375 0.6203125 0.5578125 0.63125 0.5609375
  lớp=red_cannon      số giá trị/dòng= 43  số dòng=40
     mẫu: 10 0.58125 0.4 0.575 0.4046875 0.5671875 0.4

## Làm sạch và ánh xạ về 14 lớp chuẩn
Duyệt từng bộ: đổi tên lớp về 14 lớp chuẩn, bỏ dòng `board`, bỏ ảnh có lớp `None`, chuyển polygon (super) và pose (duytranle) về bbox, kẹp box vào khung ảnh. Ảnh và nhãn sạch được gom vào /content/pool kèm manifest.csv (bộ, tập gốc, mã ảnh gốc). In bảng số box theo lớp và thống kê các thao tác đã làm.

In [ ]:
import shutil, glob, os, yaml
import pandas as pd
from collections import Counter

SPLITS = ['train', 'valid', 'test']
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
POOL = '/content/pool'

CLASSES = [f'{c}_{p}' for c in ['black', 'red']
           for p in ['advisor', 'bishop', 'cannon', 'king', 'knight', 'pawn', 'rook']]
CLASS_ID = {c: i for i, c in enumerate(CLASSES)}
PIECE_ALIAS = {'elephant': 'bishop', 'chariot': 'rook', 'general': 'king',
               'horse': 'knight', 'guard': 'advisor', 'soldier': 'pawn'}
DROP_LINES = {'board'}      # bỏ dòng nhãn, giữ ảnh
DROP_IMAGES = {'none'}      # bỏ cả ảnh có lớp này
POSE_DATASETS = {'duytranle'}   # nhãn dạng: lớp x y w h + keypoint

def load_names(yaml_path):
    with open(yaml_path, encoding='utf-8') as f:
        names = yaml.safe_load(f)['names']
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names)]
    return names

def source_id(filename):
    return filename.split('.rf.')[0]

def canon(raw):
    n = raw.strip().lower().replace('-', '_').replace(' ', '_')
    if n in DROP_LINES:
        return 'DROP_LINE'
    if n in DROP_IMAGES:
        return 'DROP_IMAGE'
    color, piece = n.split('_', 1)
    name = f'{color}_{PIECE_ALIAS.get(piece, piece)}'
    if name not in CLASS_ID:
        raise ValueError(f'Lớp lạ chưa ánh xạ: {raw}')
    return name

if os.path.exists(POOL):
    shutil.rmtree(POOL)
os.makedirs(f'{POOL}/images'); os.makedirs(f'{POOL}/labels')

rows, cls_count, stats = [], {}, {}
for name, p in paths.items():
    yf = glob.glob(f'{p}/**/data.yaml', recursive=True)[0]
    root = os.path.dirname(yf)
    names = load_names(yf)
    pose = name in POSE_DATASETS
    st, cnt = Counter(), Counter()

    for split in SPLITS:
        img_dir, lbl_dir = f'{root}/{split}/images', f'{root}/{split}/labels'
        if not os.path.isdir(lbl_dir):
            continue
        imgs = {os.path.splitext(f)[0]: f for f in os.listdir(img_dir) if f.lower().endswith(IMG_EXT)}
        for lf in os.listdir(lbl_dir):
            stem = os.path.splitext(lf)[0]
            if stem not in imgs:
                continue
            out, kept, skip = [], [], False
            for line in open(f'{lbl_dir}/{lf}'):
                parts = line.split()
                if not parts:
                    continue
                c = canon(names[int(parts[0])])
                if c == 'DROP_LINE':
                    st['dòng board bỏ'] += 1
                    continue
                if c == 'DROP_IMAGE':
                    skip = True
                    break
                vals = list(map(float, parts[1:]))
                if len(vals) == 4:
                    x, y, w, h = vals
                elif pose and len(vals) >= 4:
                    x, y, w, h = vals[:4]; st['pose→bbox'] += 1
                elif len(vals) >= 6 and len(vals) % 2 == 0:
                    xs, ys = vals[0::2], vals[1::2]
                    x0, x1, y0, y1 = min(xs), max(xs), min(ys), max(ys)
                    x, y, w, h = (x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0
                    st['polygon→bbox'] += 1
                else:
                    st['dòng lỗi bỏ'] += 1
                    continue
                x0, y0 = max(0.0, x - w / 2), max(0.0, y - h / 2)
                x1, y1 = min(1.0, x + w / 2), min(1.0, y + h / 2)
                w, h = x1 - x0, y1 - y0
                if w <= 0.005 or h <= 0.005:
                    st['box quá nhỏ/lỗi bỏ'] += 1
                    continue
                out.append(f'{CLASS_ID[c]} {(x0 + x1) / 2:.6f} {(y0 + y1) / 2:.6f} {w:.6f} {h:.6f}')
                kept.append(c)
            if skip:
                st['ảnh bỏ (lớp None)'] += 1
                continue
            if not out:
                st['ảnh bỏ (không còn box)'] += 1
                continue
            new = f'{name}__{imgs[stem]}'
            shutil.copy(f'{img_dir}/{imgs[stem]}', f'{POOL}/images/{new}')
            with open(f'{POOL}/labels/{os.path.splitext(new)[0]}.txt', 'w') as f:
                f.write('\n'.join(out) + '\n')
            cnt.update(kept)
            rows.append({'file': new, 'dataset': name, 'orig_split': split,
                         'source_id': f'{name}__{source_id(imgs[stem])}', 'n_box': len(out)})
    cls_count[name], stats[name] = cnt, st

manifest = pd.DataFrame(rows)
manifest.to_csv(f'{POOL}/manifest.csv', index=False)

print('Số ảnh sạch theo bộ và tập gốc:')
print(manifest.groupby(['dataset', 'orig_split']).size().unstack(fill_value=0))
table = pd.DataFrame({n: [c.get(k, 0) for k in CLASSES] for n, c in cls_count.items()}, index=CLASSES)
table['tong'] = table.sum(axis=1)
print('\nSố box theo lớp chuẩn:')
print(table)
print('\nThao tác đã thực hiện:')
for n, s in stats.items():
    print(f'  {n}: {dict(s)}')

Số ảnh sạch theo bộ và tập gốc:
orig_split         test  train  valid
dataset                              
duytranle           151   1202    150
new_chinese_chess   139    864    250
scara                72   1518    144
super                57   2004    257

Số box theo lớp chuẩn:
               super  scara  new_chinese_chess  duytranle   tong
black_advisor   2870   3084                 89       2462   8505
black_bishop    2465   2742                 90       2332   7629
black_cannon    1341   3253                 89       1903   6586
black_king      1629   1396                 90       1291   4406
black_knight    1921   2464                 92       1779   6256
black_pawn      5217   5949                 89       4265  15520
black_rook      2684   2951                 90       1866   7591
red_advisor     2752   3231                105       2385   8473
red_bishop      2616   2763                104       2333   7816
red_cannon      1512   2672                104       1792   6080
r

## Gom nhóm ảnh trùng (cùng ảnh gốc hoặc gần giống nhau)
Tính perceptual hash cho từng ảnh trong /content/pool rồi gom nhóm bằng union-find: ảnh cùng source_id là một nhóm; ảnh cùng bộ chỉ nối khi hash trùng khít (SAME_DS_T); ảnh khác bộ nối khi khoảng cách hash <= CROSS_DS_T. In kích thước nhóm, thành phần của nhóm lớn nhất, số nhóm nối nhiều bộ, và vẽ vài cặp ảnh trùng khác bộ để kiểm tra bằng mắt. Chưa xóa gì.

In [ ]:
import numpy as np, imagehash, matplotlib.pyplot as plt
from PIL import Image
from collections import defaultdict

SAME_DS_T = 0    # ngưỡng phash nối ảnh CÙNG bộ (0 = hash trùng khít; đặt -1 để chỉ gom theo source_id)
CROSS_DS_T = 4   # ngưỡng phash nối ảnh KHÁC bộ
manifest = pd.read_csv(f'{POOL}/manifest.csv')
n = len(manifest)

# 1) perceptual hash
hashes = []
for f in manifest['file']:
    try:
        with Image.open(f'{POOL}/images/{f}') as im:
            hashes.append(int(str(imagehash.phash(im)), 16))
    except Exception as e:
        print('Lỗi ảnh:', f, e); hashes.append(0)
H = np.array(hashes, dtype=np.uint64)
ds = manifest['dataset'].values

# 2) union-find
parent = list(range(n))
def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]
        a = parent[a]
    return a
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

first = {}
for i, s in enumerate(manifest['source_id']):
    if s in first: union(first[s], i)
    else: first[s] = i

def popcount64(x):
    return np.unpackbits(x.view(np.uint8).reshape(-1, 8), axis=1).sum(axis=1)

for i in range(n - 1):
    d = popcount64(H[i + 1:] ^ H[i])
    thr = np.where(ds[i + 1:] == ds[i], SAME_DS_T, CROSS_DS_T)
    for j in (np.nonzero(d <= thr)[0] + i + 1):
        union(i, int(j))

manifest['group'] = [find(i) for i in range(n)]

# 3) thống kê
g = manifest.groupby('group')
sizes = g.size().sort_values(ascending=False)
multi_ds = g['dataset'].nunique()
multi_sp = g['orig_split'].nunique()
print(f'Ảnh: {n} | nhóm: {len(sizes)} | nhóm lớn nhất: {sizes.head(5).tolist()}')
top = sizes.index[0]
print('Thành phần nhóm lớn nhất:', manifest[manifest.group == top].groupby(['dataset', 'orig_split']).size().to_dict())
print(f'Nhóm trải qua >1 bộ: {(multi_ds > 1).sum()} | nhóm nằm ở >1 tập gốc: {(multi_sp > 1).sum()}')
if sizes.iloc[0] > 0.03 * n:
    print('⚠ Nhóm lớn nhất quá lớn: đặt SAME_DS_T = -1 và chạy lại.')

print('\nCặp bộ có ảnh trùng nhau:')
pair = defaultdict(int)
for gid in multi_ds[multi_ds > 1].index:
    sets = tuple(sorted(manifest.loc[manifest.group == gid, 'dataset'].unique()))
    pair[sets] += 1
for k, v in sorted(pair.items(), key=lambda kv: -kv[1]):
    print(' ', k, v)

# 4) vẽ vài cặp trùng khác bộ để kiểm tra bằng mắt
cross = multi_ds[multi_ds > 1].index[:3]
if len(cross):
    fig, axes = plt.subplots(len(cross), 2, figsize=(8, 3.5 * len(cross)), squeeze=False)
    for r, gid in enumerate(cross):
        sub = manifest[manifest.group == gid].drop_duplicates('dataset').head(2)
        for c, (_, row) in enumerate(sub.iterrows()):
            axes[r][c].imshow(Image.open(f'{POOL}/images/{row.file}'))
            axes[r][c].set_title(row.dataset); axes[r][c].axis('off')
    plt.tight_layout(); plt.show()

Ảnh: 6808 | nhóm: 4265 | nhóm lớn nhất: [48, 44, 26, 26, 24]
Thành phần nhóm lớn nhất: {('scara', 'train'): 45, ('scara', 'valid'): 3}
Nhóm trải qua >1 bộ: 0 | nhóm nằm ở >1 tập gốc: 191

Cặp bộ có ảnh trùng nhau:


## Chia train/val/test theo nhóm và xuất dataset hợp nhất
Chia ~80/10/10 theo ảnh, giữ nguyên mỗi nhóm trong cùng một tập và giữ tỷ lệ các bộ trong từng tập (StratifiedGroupKFold). Kiểm tra tỷ lệ val/test của từng bộ trước; nếu lệch thì dừng, không ghi file. Nếu ổn thì copy ảnh/nhãn vào /content/merged, ghi data.yaml với 14 lớp chuẩn, in số box theo lớp ở mỗi tập, rồi nén lưu bản sao lên Drive (ghi đè merged_v1.zip cũ).

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
import yaml

MERGED = '/content/merged'
sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
fold = np.zeros(len(manifest), dtype=int)
for k, (_, te) in enumerate(sgkf.split(manifest, manifest['dataset'], manifest['group'])):
    fold[te] = k
manifest['split'] = np.where(fold == 0, 'test', np.where(fold == 1, 'val', 'train'))

# kiểm tra tỷ lệ trước khi ghi file
share = manifest.groupby(['dataset', 'split']).size().unstack(fill_value=0)
for c in ['train', 'val', 'test']:
    if c not in share: share[c] = 0
pct = share[['val', 'test']].div(share[['train', 'val', 'test']].sum(axis=1), axis=0) * 100
print('Số ảnh theo bộ và tập:'); print(share[['train', 'val', 'test']])
print('\nTỷ lệ val/test mỗi bộ (%):'); print(pct.round(1))
bad = pct[(share[['train', 'val', 'test']].sum(axis=1) >= 100) & ((pct < 5).any(axis=1) | (pct > 20).any(axis=1))]
if len(bad):
    raise ValueError(f'Chia tập lệch ở bộ: {list(bad.index)}. Xem lại bước gom nhóm (nhóm quá lớn?), chưa ghi file.')

if os.path.exists(MERGED):
    shutil.rmtree(MERGED)
for s in ['train', 'val', 'test']:
    os.makedirs(f'{MERGED}/{s}/images'); os.makedirs(f'{MERGED}/{s}/labels')

for _, r in manifest.iterrows():
    stem = os.path.splitext(r.file)[0]
    shutil.copy(f'{POOL}/images/{r.file}', f'{MERGED}/{r.split}/images/{r.file}')
    shutil.copy(f'{POOL}/labels/{stem}.txt', f'{MERGED}/{r.split}/labels/{stem}.txt')

with open(f'{MERGED}/data.yaml', 'w') as f:
    yaml.safe_dump({'path': MERGED, 'train': 'train/images', 'val': 'val/images',
                    'test': 'test/images', 'nc': len(CLASSES), 'names': CLASSES},
                   f, allow_unicode=True, sort_keys=False)
manifest.to_csv(f'{MERGED}/manifest.csv', index=False)

box = {s: Counter() for s in ['train', 'val', 'test']}
for s in box:
    for lf in glob.glob(f'{MERGED}/{s}/labels/*.txt'):
        for line in open(lf):
            if line.strip():
                box[s][CLASSES[int(line.split()[0])]] += 1
print('\nSố box theo lớp và tập:')
print(pd.DataFrame(box, index=CLASSES))

shutil.make_archive(f'{BASE}/merged/merged_v1', 'zip', MERGED)
print('\nĐã lưu', f'{BASE}/merged/merged_v1.zip')

Số ảnh theo bộ và tập:
split              train  val  test
dataset                            
duytranle           1200  146   157
new_chinese_chess   1005  124   124
scara               1423  151   160
super               1841  254   223

Tỷ lệ val/test mỗi bộ (%):
split               val  test
dataset                      
duytranle           9.7  10.4
new_chinese_chess   9.9   9.9
scara               8.7   9.2
super              11.0   9.6

Số box theo lớp và tập:
               train   val  test
black_advisor   6923   793   789
black_bishop    6207   731   691
black_cannon    5373   617   596
black_king      3573   421   412
black_knight    5105   584   567
black_pawn     12548  1571  1401
black_rook      6228   688   675
red_advisor     6896   805   772
red_bishop      6336   759   721
red_cannon      4962   580   538
red_king        3676   450   423
red_knight      5021   612   538
red_pawn       12902  1503  1402
red_rook        6219   675   655

Đã lưu /content/drive/MyDrive/da

## Kiểm tra nhãn bằng mắt
Vẽ box và tên lớp lên 2 ảnh ngẫu nhiên của mỗi bộ trong tập train của /content/merged, để chắc việc chuyển đổi nhãn (polygon/pose → bbox) và ánh xạ lớp đúng trước khi train.

In [ ]:
import cv2, matplotlib.pyplot as plt

picks = []
for dsn in ['super', 'duytranle', 'new_chinese_chess', 'scara']:
    sub = manifest[(manifest.dataset == dsn) & (manifest.split == 'train')]
    picks += sub.sample(2, random_state=7)[['file', 'dataset']].values.tolist()

fig, axes = plt.subplots(2, 4, figsize=(24, 12))

for ax, (f, dsn) in zip(axes.flatten(), picks):
    img = cv2.cvtColor(cv2.imread(f'{MERGED}/train/images/{f}'), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    thick, fs = max(2, W // 300), max(0.5, W / 1100)
    stem = os.path.splitext(f)[0]
    for line in open(f'{MERGED}/train/labels/{stem}.txt'):
        c, x, y, w, h = line.split()
        c, x, y, w, h = int(c), float(x), float(y), float(w), float(h)
        p1 = (int((x - w / 2) * W), int((y - h / 2) * H))
        p2 = (int((x + w / 2) * W), int((y + h / 2) * H))
        color = (255, 60, 60) if CLASSES[c].startswith('red') else (60, 120, 255)
        cv2.rectangle(img, p1, p2, color, thick)
        cv2.putText(img, CLASSES[c].split('_')[1][:3], (p1[0], max(p1[1] - 4, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, fs, (255, 255, 0), max(1, thick - 1))
    ax.imshow(img); ax.set_title(dsn); ax.axis('off')
plt.tight_layout(); plt.show()

Output hidden; open in https://colab.research.google.com to view.